In [1]:
from golden_data.config import (
    get_raw_data_path,
    DATA_INTERIM,
    BASE_DIR
)

from golden_data.custom_field_scan import scan_custom_fields, parse_cf_cell
from golden_data.markdown_utils import df_to_markdown
from golden_data.ai_taxonomist import analyze_custom_fields
from golden_data.data_profiling import profile_raw_data

import pandas as pd
from collections import Counter, defaultdict
import json


In [2]:
print(get_raw_data_path())
print(BASE_DIR)
print(DATA_INTERIM)

/Users/csmiller/projects/bossard-golden-data/data/raw/bossard_raw.csv
/Users/csmiller/projects/bossard-golden-data
/Users/csmiller/projects/bossard-golden-data/data/interim


In [3]:
phase1_fields = [
    "Material",
    "Finish",
    "Type",
    "Thread",
    "Length",
    "Diameter",
    "Width",
    "Height",
    "Head Style",
    "RoHS Compliant",
]

In [4]:
from golden_data.custom_field_scan import parse_cf_cell  # if you exposed it, otherwise re-define in the notebook
import json
from collections import defaultdict, Counter

raw_path = get_raw_data_path()
df = pd.read_csv(raw_path, low_memory=False)

# Filter to product rows if present
if "Item" in df.columns:
    products = df[df["Item"] == "Product"].copy()
else:
    products = df.copy()

field_value_counts = {name: Counter() for name in phase1_fields}
product_has_field = {name: 0 for name in phase1_fields}
total_products = len(products)

for _, row in products.iterrows():
    for name, value in parse_cf_cell(row.get("Custom Fields")):
        if name in phase1_fields and value:
            field_value_counts[name][value] += 1
            product_has_field[name] += 1

coverage_rows = []
for name in phase1_fields:
    cnt = product_has_field[name]
    pct = 100.0 * cnt / total_products if total_products else 0.0
    unique_vals = len(field_value_counts[name])
    coverage_rows.append(
        {
            "Field_Name": name,
            "Products_With_Value": cnt,
            "Coverage_Percent": round(pct, 2),
            "Unique_Values": unique_vals,
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.sort_values("Coverage_Percent", ascending=False)

,Field_Name,Products_With_Value,Coverage_Percent,Unique_Values
0,Material,6178,6.94,205
2,Type,3994,4.49,1609
1,Finish,3557,4.00,137
3,Thread,3223,3.62,204
4,Length,2937,3.30,1098
9,RoHS Compliant,1381,1.55,7
6,Width,1024,1.15,352
7,Height,832,0.93,319
5,Diameter,806,0.91,257
8,Head Style,636,0.71,72


In [5]:
from golden_data.config import get_raw_data_path
from golden_data.custom_field_scan import parse_cf_cell
from collections import Counter, defaultdict
import pandas as pd

# Define your Phase 1 attributes
phase1_fields = [
    "Material",
    "Finish",
    "Type",
    "Thread",
    "Length",
    "Diameter",
    "Width",
    "Height",
    "Head Style",
    "RoHS Compliant",
]

raw_path = get_raw_data_path()
df = pd.read_csv(raw_path, low_memory=False)

# Focus on product rows if column exists
if "Item" in df.columns:
    products = df[df["Item"] == "Product"].copy()
else:
    products = df.copy()

total_products = len(products)

field_value_counts = {name: Counter() for name in phase1_fields}
product_has_field = {name: 0 for name in phase1_fields}

for _, row in products.iterrows():
    for name, value in parse_cf_cell(row.get("Custom Fields")):
        if name in phase1_fields and value:
            field_value_counts[name][value] += 1
            product_has_field[name] += 1

coverage_rows = []
for name in phase1_fields:
    cnt = product_has_field[name]
    pct = 100.0 * cnt / total_products if total_products else 0.0
    unique_vals = len(field_value_counts[name])
    coverage_rows.append(
        {
            "Field_Name": name,
            "Products_With_Value": cnt,
            "Coverage_Percent": round(pct, 2),
            "Unique_Values": unique_vals,
        }
    )

coverage_df = pd.DataFrame(coverage_rows).sort_values("Coverage_Percent", ascending=False)
coverage_df

,Field_Name,Products_With_Value,Coverage_Percent,Unique_Values
0,Material,6178,6.94,205
2,Type,3994,4.49,1609
1,Finish,3557,4.00,137
3,Thread,3223,3.62,204
4,Length,2937,3.30,1098
9,RoHS Compliant,1381,1.55,7
6,Width,1024,1.15,352
7,Height,832,0.93,319
5,Diameter,806,0.91,257
8,Head Style,636,0.71,72


In [6]:
def show_top_values(field_name, top_n=30):
    cnt = field_value_counts[field_name]
    top = cnt.most_common(top_n)
    return pd.DataFrame(top, columns=[f"{field_name}_Value", "Count"])

material_top = show_top_values("Material", top_n=50)
finish_top = show_top_values("Finish", top_n=50)

material_top.head(20), finish_top.head(20)

material_top.to_csv(DATA_INTERIM / "material_top_values.csv", index=False)
finish_top.to_csv(DATA_INTERIM / "finish_top_values.csv", index=False)


In [7]:
import re

length_values = [v for v, _ in field_value_counts["Length"].most_common(300)]
length_sample_df = pd.DataFrame({"Raw_Length": length_values})

def classify_length_pattern(s: str) -> str:
    s = s.strip()
    if re.search(r"\bmm\b", s, re.IGNORECASE):
        return "mm"
    if re.search(r"\b(in\.?|inch(?:es)?)\b", s, re.IGNORECASE):
        return "inch"
    if re.search(r"\bto\b|\-", s):
        return "range"
    if re.match(r"^[0-9\.\s/]+$", s):
        return "numeric_only"
    return "other"

length_sample_df["Pattern"] = length_sample_df["Raw_Length"].apply(classify_length_pattern)
length_sample_df["Pattern"].value_counts()


Pattern
inch            205
mm               77
other            14
numeric_only      4
Name: count, dtype: int64

In [8]:
for pattern in ["mm", "inch", "range", "numeric_only", "other"]:
    print(f"\n=== Pattern: {pattern} ===")
    display(length_sample_df[length_sample_df["Pattern"] == pattern].head(10))


=== Pattern: mm ===


,Raw_Length,Pattern
4,5.90 mm.,mm
6,4.60 mm.,mm
9,6.5 mm.,mm
10,8.2 mm.,mm
33,9.6 mm.,mm
35,6 mm.,mm
43,10 mm.,mm
58,5 mm.,mm
65,0.75 mm.,mm
70,12 mm.,mm



=== Pattern: inch ===


,Raw_Length,Pattern
0,.256 in.,inch
2,.206 in.,inch
3,.176 in.,inch
5,.237 in.,inch
7,1.000 in.,inch
8,.337 in.,inch
11,12 in.,inch
12,0.375 in.,inch
13,.250 in.,inch
14,20 in.,inch



=== Pattern: range ===


,Raw_Length,Pattern



=== Pattern: numeric_only ===


,Raw_Length,Pattern
112,10,numeric_only
144,20,numeric_only
240,5,numeric_only
241,16,numeric_only



=== Pattern: other ===


,Raw_Length,Pattern
1,6.000 ft.,other
24,100.000 ft.,other
29,100 ft (30.5 m),other
75,100.000 ft. Reel,other
77,100.000 ft. per Reel,other
99,36 yd.,other
140,6.600 ft.,other
149,72 yd.,other
155,4 ft (1.2 m),other
198,60 yd.,other


In [9]:
length_sample_df.to_csv(DATA_INTERIM / "length_value_patterns.csv", index=False)

In [10]:
from golden_data.scripts.demo_scan import analysis_text, cf_summary
from golden_data.config import BASE_DIR
import textwrap

reports_dir = BASE_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

report_path = reports_dir / "attributes_report.md"

# Top 20 custom fields (from cf_summary)
top_fields_md = df_to_markdown(cf_summary.head(20))

# Phase-1 coverage table
coverage_md = df_to_markdown(coverage_df)

# Short Material / Finish sample tables for stakeholders
material_md = df_to_markdown(material_top.head(15))
finish_md = df_to_markdown(finish_top.head(15))

wrapped_analysis = textwrap.dedent(analysis_text).strip()

report_md = f"""# Bossard BigCommerce Attribute Discovery Report

**Prepared by:** Golden Data Toolkit  
**Data Source:** {raw_path.name}  
**Records Analyzed:** {len(df)} products (raw export)  
**Custom Field Sample:** First 5,000 product rows

---

## 1. Objective

This analysis reviews the current BigCommerce custom field structure and attribute usage
to identify:

- Gaps and inconsistencies in product attributes
- High-value attributes for normalization and taxonomy design
- Priority steps to enable Bossard’s 2026 North American marketing objectives:

1. **Brand Expansion Across North America**  
2. **Industry-Specific Demand Generation**  
3. **Digital Excellence & Marketing Automation**  
4. **Tight Sales Alignment With Measurable Revenue Impact**

---

## 2. Data Validation Summary

- **File path:** `{raw_path}`
- **Total rows in export:** {len(df)}
- **Total columns in export:** {len(df.columns)}
- **Contains 'Custom Fields' column:** {"Yes" if "Custom Fields" in df.columns else "No"}
- **Contains 'Item' filter column:** {"Yes" if "Item" in df.columns else "No"}

A subset of up to 5,000 product rows was used for custom field analysis.

---

## 3. Top Custom Attributes (Usage Snapshot)

The table below shows the most frequently used custom fields across the analyzed products:

{top_fields_md}

---

## 4. Phase-1 Attribute Coverage

The following attributes are proposed as **Phase 1** for normalization:

- Material, Finish, Type, Thread  
- Length, Diameter, Width, Height  
- Head Style, RoHS Compliant

Coverage across the catalog:

{coverage_md}

**Interpretation:**

- High-coverage fields are ideal for first-wave normalization because improvements will benefit a large portion of the catalog.
- Fields with many unique values (especially Material and Finish) are strong candidates for mapping tables and controlled vocabularies.

---

## 5. Sample Raw Values (Material & Finish)

These tables illustrate the kind of raw values that will be normalized.

### 5.1 Material (top examples)

{material_md}

### 5.2 Finish (top examples)

{finish_md}

These samples highlight:

- Synonymous or near-duplicate values (e.g., “Stainless”, “Stainless Steel”, “SS”)
- Supplier-specific or overly detailed strings that should be mapped to clean, customer-facing terms
- Spelling or formatting inconsistencies that will be addressed in the golden dataset

---

## 6. AI Taxonomist Analysis

The following analysis was generated by an AI “senior taxonomist” agent tuned to Bossard’s 2026 strategy.
It reviews attribute quality, severity of issues, and recommends a roadmap for normalization and governance.

{wrapped_analysis}

---

## 7. Recommended Normalization Roadmap

Based on the attribute scan and AI analysis, the following **Phase 1** attributes are recommended
for normalization and governance:

- **Core dimensional attributes:** Length, Width, Height, Diameter, Grip Range  
- **Core material & finish attributes:** Material, Finish, Plating/Coating  
- **Fastener identity attributes:** Type, Head Style, Thread, Size/Class  
- **Compliance & restrictions:** RoHS Compliant, Access Restriction  

(Additional narrative can stay as previously drafted, or be lightly edited here.)

---

## 8. Actions for Bossard Marketing Excellence

(Reuse or adapt the earlier “Actions aligned to the four pillars” section – your previous text already fits nicely.)

---

## 9. Next Steps

1. Approve Phase 1 attribute scope and normalization approach.
2. Build mapping tables for Material and Finish (seeded from the exported `*_top_values.csv` files).
3. Design and implement parsing & unit standardization rules for Length and other dimensional fields.
4. Validate the golden attribute set on a targeted subset (e.g., one supplier, one fastener family).
5. Deploy normalized attributes into BigCommerce and review SEO, navigation, and AI search behavior.
6. Formalize governance so new products are onboarded with normalized attributes from day one.
"""

report_path.write_text(report_md, encoding="utf-8")
report_path

   Field_Name  Product_Count  Usage_Percent  \
0    Material           2479          49.58   
1  Spec Sheet           1975          39.50   
2        Type           1506          30.12   
3      Length           1437          28.74   
4       Color           1297          25.94   

                                      Example_Values  
0  Steel, Soft Urethane, Nylon 6/6, Polypropylene...  
1  <a href=/content/ProductSpecSheets/FCIAMPE0000...  
2  Single Row, Vertical, Flexible, Light Weight, ...  
3         60 in., 8.000  in., 11 in., 12 in., 75 in.  
4                Black, Natural, Bright, White, Gray  
To analyze the provided dataset effectively, we will focus on the following goals:

1. **Identify Key Attributes**: Determine which attributes are most significant for product categorization and customer decision-making.
2. **Assess Data Completeness**: Evaluate the completeness of the data for each attribute and identify any gaps.
3. **Understand Usage Trends**: Analyze the usage per

PosixPath('/Users/csmiller/projects/bossard-golden-data/reports/attributes_report.md')

In [11]:
from pathlib import Path
from golden_data.config import DATA_INTERIM
from golden_data.normalization import create_mapping_template_from_top_values, get_mapping_path

material_top_path = DATA_INTERIM / "material_top_values.csv"
finish_top_path = DATA_INTERIM / "finish_top_values.csv"

material_mapping_path = create_mapping_template_from_top_values(
    material_top_path,
    field_name="Material",
    normalized_column="normalized_material",
    overwrite=False,
)

finish_mapping_path = create_mapping_template_from_top_values(
    finish_top_path,
    field_name="Finish",
    normalized_column="normalized_finish",
    overwrite=False,
)

material_mapping_path, finish_mapping_path

(PosixPath('/Users/csmiller/projects/bossard-golden-data/mappings/material_mapping.csv'),
 PosixPath('/Users/csmiller/projects/bossard-golden-data/mappings/finish_mapping.csv'))